### Split by characters

In [ ]:
import regex as re

def split_chars(text):
    return re.findall(r".", text)

In [ ]:
chars = split_chars("The quick brown fox jumped over the lazy dog.")

In [ ]:
chars[:12]

['T', 'h', 'e', ' ', 'q', 'u', 'i', 'c', 'k', ' ', 'b', 'r']

### Split by words

In [ ]:
def split_words(text):
    return re.findall(r"[\w]+|[.,!]", text)

In [ ]:
words = split_words("The quick brown fox jumped over the lazy dog.")

In [ ]:
words

['The', 'quick', 'brown', 'fox', 'jumped', 'over', 'the', 'lazy', 'dog', '.']

### Build a vocabulary

In [ ]:
vocabulary = {
    "[UNK]": 0,
    "the": 1,
    "quick": 2,
    "brown": 3,
    "fox": 4,
    "jumped": 5,
    "over": 6,
    "dog": 7,
    ".": 8
}

In [ ]:
words = split_words("The quick brown fox jumped over the lazy dog.")
indices = [vocabulary.get(word, 0) for word in words]

In [ ]:
indices

[0, 2, 3, 4, 5, 6, 1, 0, 7, 8]

### Standardization

- Standardization - Normalize input with the basic yexy-to-text transformations

- Splitting - We split our text into sequences of tokens

- Indexing - Map our tokens to indices using a vocabulary.

### Stemming or lemmatization

Map words to a common root (eg, "tired" -> "tire" and "trophies" -> tropy)

### Tokenization

- Standardization
- Tokenization

### Character tokenization

In [ ]:
class CharTokenizer:
    def __init__(self, vocabulary):
        self.vocabulary = vocabulary
        self.unk_id = vocabulary["[UNK]"]

    def standardize(self, inputs):
        return inputs.lower()

    def split(self, inputs):
        return re.split(r".", inputs)

    def index(self, inputs):
        return [self.vocabulary.get(word, self.unk_id) for word in inputs]

    def __call__(self, inputs):
        inputs = self.standardize(inputs)
        tokens = self.split(inputs)
        indices = self.index(tokens)
        return indices

In [ ]:
import collections

def compute_char_vocabulary(inputs, max_size):
    char_counts = collections.Counter()
    for x in inputs:
        x = x.lower()
        tokens = re.findall(r".", x)
        char_counts.update(tokens)
    vocabulary = ["[UNK]"]
    most_common = char_counts.most_common(max_size - len(vocabulary))
    for token, count in most_common:
        vocabulary.append(token)

    return dict((token, i) for i, token in enumerate(vocabulary))
    # return char_counts

In [ ]:
compute_char_vocabulary("The quick brown fox jumped over the lazy dog.", 100)

{'[UNK]': 0,
 ' ': 1,
 'e': 2,
 'o': 3,
 't': 4,
 'h': 5,
 'u': 6,
 'r': 7,
 'd': 8,
 'q': 9,
 'i': 10,
 'c': 11,
 'k': 12,
 'b': 13,
 'w': 14,
 'n': 15,
 'f': 16,
 'x': 17,
 'j': 18,
 'm': 19,
 'p': 20,
 'v': 21,
 'l': 22,
 'a': 23,
 'z': 24,
 'y': 25,
 'g': 26,
 '.': 27}

In [ ]:
import keras

filename = keras.utils.get_file(
    origin="https://www.gutenberg.org/files/2701/2701-0.txt",
)
moby_dick = list(open(filename, "r"))

vocabulary = compute_char_vocabulary(moby_dick, max_size=100)
char_tokenizer = CharTokenizer(vocabulary)

In [ ]:
print("Vocabulary length: ", len(vocabulary))

Vocabulary length:  74


In [ ]:
print("Vocabulary start:", list(vocabulary.keys())[:10])

Vocabulary start: ['[UNK]', ' ', 'e', 't', 'a', 'o', 'n', 'i', 's', 'h']


In [ ]:
print("Vocabulary end: ", list(vocabulary.keys())[-10:])

Vocabulary end:  ['ח', 'ו', '\u200e', 'ϰ', 'η', 'τ', 'ο', 'ς', "'", 'â']


In [ ]:
class WordTokenizer:
    def __init__(self, vocabulary):
        self.vocabulary = vocabulary
        self.unk_id = vocabulary["[UNK]"]

    def standardize(self, inputs):
        return inputs.lower()

    def split(self, inputs):
        return re.findall(r"[\w]+|[.,!?;]", inputs)

    def index(self, tokens):
        return [self.vocabulary.get(t, self.unk_id) for t in tokens]

    def __call__(self, inputs):
        inputs = self.standardize(inputs)
        tokens = self.split(inputs)
        indices = self.index(tokens)
        return indices

In [ ]:
def compute_word_vocabulary(inputs, max_size):
    word_counts = collections.Counter()
    for x in inputs:
        x = x.lower()
        tokens = re.findall(r"[\w]+|[.,!?;]", x)
        word_counts.update(tokens)
    vocabulary = ["[UNK]"]
    most_common = word_counts.most_common(max_size - len(vocabulary))
    for token, count in most_common:
        vocabulary.append(token)
    return dict((token, i) for i, token in enumerate(vocabulary))

In [ ]:
vocabulary = compute_word_vocabulary(moby_dick, max_size=2_000)
word_tokenizer = WordTokenizer(vocabulary)

In [ ]:
print("Vocabulary length:", len(vocabulary))

Vocabulary length: 2000


In [ ]:
print("Vocabulary start:", list(vocabulary.keys())[:5])

Vocabulary start: ['[UNK]', ',', 'the', '.', 'of']


In [ ]:
print("Vocabulary end:", list(vocabulary.keys())[-5:])

Vocabulary end: ['beholding', 'hearse', 'mount', 'tackle', 'objects']


In [ ]:
print("Line length:", len(word_tokenizer("Call me Ishmael. Some years ago--never mind how long precisely.")))

Line length: 13


In [ ]:
word_tokenizer("Call me Ishmael. Some years ago--never mind how long precisely.")

[407, 47, 1021, 3, 49, 248, 607, 145, 296, 117, 87, 795, 3]

## Byte-pair encoding

In [ ]:
data = [
    "the quick brown fox",
    "the slow brown fox",
    "the quick brown foxhound",
]

In [ ]:
def count_and_split_words(data):
    counts = collections.Counter()
    for line in data:
        line = line.lower()
        for word in re.findall(r"[\w]+|[.,!?;]", line):
            chars = re.findall(r".", word)
            split_word = " ".join(chars)
            counts[split_word] += 1
    return dict(counts)

counts = count_and_split_words(data)

In [ ]:
counts

{'t h e': 3,
 'q u i c k': 2,
 'b r o w n': 3,
 'f o x': 2,
 's l o w': 1,
 'f o x h o u n d': 1}

In [ ]:
def count_pairs(counts):
    pairs = collections.Counter()
    for word, freq in counts.items():
        symbols = word.split()
        for pair in zip(symbols[:-1], symbols[1:]):
            pairs[pair] += freq
    return pairs

pairs = count_pairs(counts)

In [ ]:
pairs

Counter({('t', 'h'): 3,
         ('h', 'e'): 3,
         ('q', 'u'): 2,
         ('u', 'i'): 2,
         ('i', 'c'): 2,
         ('c', 'k'): 2,
         ('b', 'r'): 3,
         ('r', 'o'): 3,
         ('o', 'w'): 4,
         ('w', 'n'): 3,
         ('f', 'o'): 3,
         ('o', 'x'): 3,
         ('s', 'l'): 1,
         ('l', 'o'): 1,
         ('x', 'h'): 1,
         ('h', 'o'): 1,
         ('o', 'u'): 1,
         ('u', 'n'): 1,
         ('n', 'd'): 1})

In [ ]:
def count_pairs(counts):
    pairs = collections.Counter()
    for word, freq in counts.items():
        symbols = word.split()
        for pair in zip(symbols[:-1], symbols[1:]):
            pairs[pair] += freq
    return pairs

def merge_pair(counts, first, second):
    # Matches an unmerged pair
    split = re.compile(f"(?<!\S){first} {second}(?!\S)")
    # Replaces all occurances with a merged version
    merged = f"{first}{second}"
    return {split.sub(merged, word): count for word, count in counts.items()}

for i in range(10):
    pairs = count_pairs(counts)
    first, second = max(pairs, key=pairs.get)
    counts = merge_pair(counts, first, second)
    print(list(counts.keys()))

['t h e', 'q u i c k', 'b r ow n', 'f o x', 's l ow', 'f o x h o u n d']
['th e', 'q u i c k', 'b r ow n', 'f o x', 's l ow', 'f o x h o u n d']
['the', 'q u i c k', 'b r ow n', 'f o x', 's l ow', 'f o x h o u n d']
['the', 'q u i c k', 'br ow n', 'f o x', 's l ow', 'f o x h o u n d']
['the', 'q u i c k', 'brow n', 'f o x', 's l ow', 'f o x h o u n d']
['the', 'q u i c k', 'brown', 'f o x', 's l ow', 'f o x h o u n d']
['the', 'q u i c k', 'brown', 'fo x', 's l ow', 'fo x h o u n d']
['the', 'q u i c k', 'brown', 'fox', 's l ow', 'fox h o u n d']
['the', 'qu i c k', 'brown', 'fox', 's l ow', 'fox h o u n d']
['the', 'qui c k', 'brown', 'fox', 's l ow', 'fox h o u n d']


<>:11: SyntaxWarning: invalid escape sequence '\S'
<>:11: SyntaxWarning: invalid escape sequence '\S'
<>:11: SyntaxWarning: invalid escape sequence '\S'
<>:11: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_817/144060027.py:11: SyntaxWarning: invalid escape sequence '\S'
  split = re.compile(f"(?<!\S){first} {second}(?!\S)")
/tmp/ipykernel_817/144060027.py:11: SyntaxWarning: invalid escape sequence '\S'
  split = re.compile(f"(?<!\S){first} {second}(?!\S)")
